# 2. Quality Control and Denoising
In this notebook, we examined the quality of our sequences, performed trimming and filtering to remove low-quality reads, applied DADA2 to denoise the data, merge our paired-end reads and identify amplicon sequence variants (ASVs) in the end. 
### Import Packages

In [1]:
# Import all necessary packages
import IPython
import pandas as pd
import matplotlib.pyplot as plt
import os
import qiime2 as q2
from qiime2 import Visualization

%matplotlib inline

### Set Working Directory
Ensure that the working directory is correctly set to the 'scripts' folder within the main project directory. 
Otherwise, the file paths used in this notebook may not work properly.

In [2]:
# The working directory should normally default to the 'scripts' folder. 
# If it doesn't, set it manually using the command below.
# os.chdir("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts")  # Adjust this path to match your folder structure.

# Verify that your working directory is the 'scripts' folder inside the main project directory (.../MicrobiomeAnalysis_TummyTribe/scripts)
cwd = os.getcwd()
if not cwd.endswith("MicrobiomeAnalysis_TummyTribe/scripts"):
    print("WARNING: The working directory is not set to the 'scripts' folder inside 'MicrobiomeAnalysis_TummyTribe'!")
    print("Current working directory:", cwd)
    print("Please set it manually using os.chdir().")
else:
    print(f"Working directory is correctly set to the 'scripts' folder (\"{cwd}\").")

Working directory is correctly set to the 'scripts' folder ("/home/jovyan/MicrobiomeAnalysis_TummyTribe/scripts").


In [3]:
# Data directories
data_dir = "../data/raw"
denoised_data_dir = "../data/processed/denoising"

## Quality Control
First, we assessed the quality of our sequencing data to determine the necessary preprocessing steps.

In [4]:
! qiime demux summarize \
    --i-data $data_dir/sequences-demux-paired.qza \
    --o-visualization $data_dir/sequences-demux-paired.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/raw/sequences-demux-paired.qzv


In [4]:
Visualization.load(f"{data_dir}/sequences-demux-paired.qzv")

<visualization: Visualization uuid: 94829b8b-c8ec-4e15-90c0-4279dbfdcc0f>

The number of sequences per sample appears typical for microbiome data. However, the interactive quality plot clearly shows that our data is somewhat older. The median quality of the forward reads is fairly good, although some whiskers drop considerably. The quality of the reverse reads, on the other hand, is quite poor. Even at the beginning of the reads, the lower quartile quality drops below 20, and after approximately 150 nucleotides, the quality decreases further, with many sequences falling below a score of 5. This indicates that a substantial amount of truncation is necessary.

### Denoising and merging
Since the quality of our data is not very high, preprocessing is especially important. We performed denoising using the DADA2 plugin in QIIME 2.

We tested several truncation lengths (see Additional Notebooks) and found the best results with a truncation length of 135 for both forward and reverse reads. With a lower truncation length of 130, very few sequences were retained because most could no longer be merged. The 16S rRNA V4 region is approximately 254 bp in most microbial species, so an overlap of only 6 nucleotides is insufficient for merging from both ends. DADA2 requires a minimum overlap of 12 nucleotides. With a truncation length of 135, merging is still possible.

Initially, we tried higher truncation lengths, but this resulted in too many errors, leading to very low sequence retention. For example, using truncation lengths of 170 for the forward reads and 150 for the reverse or 150 for both, some samples retained less than 5% of the sequences. By contrast, a truncation length of 135 for both reads retained at least 19% of sequences for all samples, and the number of unique sequences increased compared to higher truncation lengths.

All other parameters were kept at their default values. Reads with more than two expected errors were filtered out, sequences were merged and chimeric sequences were removed. This process produced a feature table containing amplicon sequence variant (ASV) counts per sample.

In [6]:
# DADA2 filtering, denoising, merging and chimera removal to create a feature table of ASVs
! qiime dada2 denoise-paired \
    --i-demultiplexed-seqs $data_dir/sequences-demux-paired.qza \
    --p-trunc-len-f 135 \
    --p-trunc-len-r 135 \
    --p-n-threads 3 \
    --o-table $denoised_data_dir/dada2_table.qza \
    --o-representative-sequences $denoised_data_dir/dada2_rep_seq.qza \
    --o-denoising-stats $denoised_data_dir/dada2_stats.qza \
    --verbose

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Running external command line application(s). This may print messages to stdout and/or stderr.
The command(s) being run are below. These commands cannot be manually re-run as they will depend on temporary files that no longer exist.

Command: run_dada.R --input_directory /tmp/tmpt_3_6pz4/forward --input_directory_reverse /tmp/tmpt_3_6pz4/reverse --output_path /tmp/tmpt_3_6pz4/output.tsv.biom --output_track /tmp/tmpt_3_6pz4/track.tsv --filtered_directory /tmp/tmpt_3_6pz4/filt_f --filtered_directory_reverse /tmp/tmpt_3_6pz4/filt_r --truncation_length 135 --truncation_length_reverse 135 --trim_left 0 --trim_left_reverse 0 --max_expected_errors 2.0 --max_expected_erro

In [7]:
# Visualize the resulting sequences
! qiime metadata tabulate \
    --m-input-file $denoised_data_dir/dada2_stats.qza \
    --o-visualization $denoised_data_dir/dada2_stats.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/denoising/dada2_stats.qzv


In [5]:
Visualization.load(f"{denoised_data_dir}/dada2_stats.qzv")

<visualization: Visualization uuid: 4194032e-8965-4678-bdb8-6265c5e6c77b>

In [9]:
! qiime feature-table summarize \
    --i-table $denoised_data_dir/dada2_table.qza \
    --m-sample-metadata-file $data_dir/metadata.tsv \
    --o-visualization $denoised_data_dir/dada2_table.qzv

/opt/conda/lib/python3.10/site-packages/unifrac/__init__.py:9: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
Saved Visualization to: ../data/processed/denoising/dada2_table.qzv


In [6]:
Visualization.load(f"{denoised_data_dir}/dada2_table.qzv")

<visualization: Visualization uuid: 8ab0c37e-c3a6-42c9-9f98-cee916c6d746>

We end up with 3'873 unique features (number of unique ASVs) and roughly 16 million reads. So we have overall good microbial richness and sequencing depth. 